In [29]:
from __future__ import annotations

import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [30]:
class Task(BaseModel): #inheriting from pydantic base model
    id: int
    title: str
    brief: str = Field(...,description="What to cover") #no default value

In [31]:
class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

In [32]:
class State(TypedDict): #Dictionary with this type of keys and val
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str

In [33]:
# from dotenv import load_dotenv
# import os
# load_dotenv()
llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
def orchestrator(state: State) -> dict :
    plan = llm.with_structured_output(Plan).invoke(
        [
            SystemMessage(
                content=("Create a blog plan with 5-7 sections on the following topic.")
            ),
            HumanMessage(
                content=(f"Topic:{state['topic']}")
            )
        ]
    )
    return {"plan": plan}

In [35]:
def fanout(state: State):
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]